In [ ]:
# testtttt

# Bayesian Optimization mit globalen Nullmodellen aus Ordner mit counter
# Sucht nach der besten Parameterkombination für die Proximity-Funktion
# unter Verwendung von Bayesian Optimization.

# ================================
# Imports
# ================================
import os
import numpy as np
import pandas as pd
from skopt import gp_minimize
from skopt.space import Real, Integer
import matplotlib.pyplot as plt
import seaborn as sns
import time

# ================================
# Proximity-Funktion
# ================================

def proximity_max_freq(
    x,
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    freqs = np.arange(1, max_freq + 1)
    all_freq_values = []

    for freq in freqs:
        weight = 1.0 / (freq ** weight_exponent)
        freq_sharpness_base = sharpness_base * (freq ** freq_sharpness_exp)
        freq_sharpness_growth = sharpness_growth * (freq ** freq_sharpness_exp)

        sharpness = freq_sharpness_base * (x ** freq_sharpness_growth)
        cos_term = ((1 + np.cos(2 * np.pi * freq * x)) / 2) ** sharpness

        peak_sharpness = freq_sharpness_base * (norm_x ** freq_sharpness_growth)
        peak_value = ((1 + np.cos(2 * np.pi * freq * norm_x)) / 2) ** peak_sharpness
        peak_value = np.where(peak_value == 0, 1e-12, peak_value)

        denom = np.where(x == 0, 1e-12, x ** decay)
        val = weight * (cos_term / peak_value) / denom
        all_freq_values.append(val)

    all_freq_values = np.array(all_freq_values)
    proximity_max = np.max(all_freq_values, axis=0)

    max_at_norm_x = np.max([
        (1.0 / (freq ** weight_exponent)) * 1.0 / (norm_x ** decay)
        for freq in freqs
    ])
    proximity_max = proximity_max / max_at_norm_x * output_max
    return proximity_max

# ================================
# Folder-Analyse
# ================================
_already_printed_global_ioi = False

def analyze_folder_global_null(folder_path, params, num_sequences=500, seed=42):
    global _already_printed_global_ioi
    rng = np.random.default_rng(seed)
    results_list = []

    # -----------------------------
    # 1) Alle IOIs laden und globale Parameter bestimmen
    # -----------------------------
    all_iois = []
    seqs = []

    for fname in os.listdir(folder_path):
        if not fname.endswith(".csv"):
            continue

        df = pd.read_csv(os.path.join(folder_path, fname))
        if "IOI" not in df.columns:
            continue

        iois = df["IOI"].dropna().values
        if len(iois) < 3:
            continue

        all_iois.extend(iois)
        seqs.append((fname, iois))

    all_iois = np.array(all_iois)
    global_median_ioi = np.median(all_iois)
    global_min = np.percentile(all_iois, 5)
    global_max = np.percentile(all_iois, 95)

    # nur beim allerersten Funktionsaufruf ausgeben
    if not _already_printed_global_ioi:
        print(f"Global median IOI = {global_median_ioi:.3f}")
        print(f"Uniform bounds = [{global_min:.3f}, {global_max:.3f}]")
        _already_printed_global_ioi = True
    
        # -----------------------------
    # 2) Nullmodelle vorbereiten (pro Sequenz-Länge!)
    # -----------------------------
    null_cache_expon = {}
    null_cache_uniform = {}
    null_cache_empirical = {}

    for fname, iois in seqs:
        n = len(iois)

        # Exponential Null
        if n not in null_cache_expon:
            medians_expon = []
            for _ in range(num_sequences):
                sim_iois = rng.exponential(scale=global_median_ioi, size=n)
                ratios = np.array([[max(sim_iois[i], sim_iois[j]) / min(sim_iois[i], sim_iois[j])
                                    for j in range(n)] for i in range(n)])
                prox_mat = proximity_max_freq(ratios, **params)
                triu_idx = np.triu_indices_from(prox_mat, k=1)
                medians_expon.append(np.median(prox_mat[triu_idx]))
            null_cache_expon[n] = np.array(medians_expon)

        # Uniform Null
        if n not in null_cache_uniform:
            medians_uniform = []
            for _ in range(num_sequences):
                sim_iois = rng.uniform(global_min, global_max, size=n)
                ratios = np.array([[max(sim_iois[i], sim_iois[j]) / min(sim_iois[i], sim_iois[j])
                                    for j in range(n)] for i in range(n)])
                prox_mat = proximity_max_freq(ratios, **params)
                triu_idx = np.triu_indices_from(prox_mat, k=1)
                medians_uniform.append(np.median(prox_mat[triu_idx]))
            null_cache_uniform[n] = np.array(medians_uniform)

        # Empirical Null (zieht zufällig IOIs aus ALLEN IOIs des Ordners)
        if n not in null_cache_empirical:
            medians_empirical = []
            for _ in range(num_sequences):
                sim_iois = rng.choice(all_iois, size=n, replace=True)
                ratios = np.array([[max(sim_iois[i], sim_iois[j]) / min(sim_iois[i], sim_iois[j])
                                    for j in range(n)] for i in range(n)])
                prox_mat = proximity_max_freq(ratios, **params)
                triu_idx = np.triu_indices_from(prox_mat, k=1)
                medians_empirical.append(np.median(prox_mat[triu_idx]))
            null_cache_empirical[n] = np.array(medians_empirical)

    # -----------------------------
    # 3) Analyse jeder Sequenz mit den globalen Nullmodellen
    # -----------------------------
    for fname, iois in seqs:
        n = len(iois)
        ratios_real = np.array([[max(iois[i], iois[j]) / min(iois[i], iois[j])
                                 for j in range(n)] for i in range(n)])
        prox_mat_real = proximity_max_freq(ratios_real, **params)
        triu_idx = np.triu_indices_from(prox_mat_real, k=1)
        proximity_real = prox_mat_real[triu_idx]
        median_real = np.median(proximity_real)

        # Exponential
        medians_expon = null_cache_expon[n]
        mean_expon = np.mean(medians_expon)
        std_expon = np.std(medians_expon, ddof=1)
        zscore_expon = (median_real - mean_expon) / std_expon if std_expon > 0 else np.nan
        pval_expon = (np.sum(medians_expon >= median_real) + 1) / (len(medians_expon) + 1)

        # Uniform
        medians_uniform = null_cache_uniform[n]
        mean_uniform = np.mean(medians_uniform)
        std_uniform = np.std(medians_uniform, ddof=1)
        zscore_uniform = (median_real - mean_uniform) / std_uniform if std_uniform > 0 else np.nan
        pval_uniform = (np.sum(medians_uniform >= median_real) + 1) / (len(medians_uniform) + 1)

        # Empirical
        medians_empirical = null_cache_empirical[n]
        mean_empirical = np.mean(medians_empirical)
        std_empirical = np.std(medians_empirical, ddof=1)
        zscore_empirical = (median_real - mean_empirical) / std_empirical if std_empirical > 0 else np.nan
        pval_empirical = (np.sum(medians_empirical >= median_real) + 1) / (len(medians_empirical) + 1)

        results_list.append({
            "filename": fname,
            "real_median": median_real,
            "expon_mean": mean_expon,
            "expon_std": std_expon,
            "uniform_mean": mean_uniform,
            "uniform_std": std_uniform,
            "empirical_mean": mean_empirical,   
            "empirical_std": std_empirical,     
            "diff_real_expon": median_real - mean_expon,
            "diff_real_uniform": median_real - mean_uniform,
            "diff_real_empirical": median_real - mean_empirical,   
            "zscore_expon": zscore_expon,
            "pval_expon": pval_expon,
            "zscore_uniform": zscore_uniform,
            "pval_uniform": pval_uniform,
            "zscore_empirical": zscore_empirical,   
            "pval_empirical": pval_empirical        
        })


    return pd.DataFrame(results_list)

# ================================
# Bayesian Optimization
# ================================

def bayesian_search_global_null(
    folder_path,
    n_calls=100,
    num_sequences=500,
    seed=42,
    objective="zscore",   # one of: "diff", "zscore", "pval"
    null_model="expon",   # name used in analyze_folder columns for now: "expon", "uniform", "empirical"
    param_space_def=None   # optional: override default param space dictionary (same keys as before)
):
    """
    Bayesian search over the proximity-function parameter space.
    objective: "diff" (max), "zscore" (max), "pval" (min)
    null_model: string used to select columns returned by analyze_folder, e.g. "expon" -> diff_real_expon, zscore_expon, pval_expon
    Returns: summary_df (all evaluated params saved), best_params (according to chosen objective)
    """

    rng = np.random.default_rng(seed)
    default_params = {"norm_x": 1.0, "output_max": 1.0}

    # default parameter-space (same ranges as before)
    if param_space_def is None:
        param_space_def = {
            "sharpness_base": (1.0, 50.0),
            "sharpness_growth": (0.0, 5.0),
            "decay": (0.0, 4.0),
            "max_freq": (1, 6),  # integer
            "weight_exponent": (0.0, 4.0),
            "freq_sharpness_exp": (0.0, 4.0)
        }

    # build skopt space
    sk_space = [
        Real(param_space_def["sharpness_base"][0], param_space_def["sharpness_base"][1], name="sharpness_base"),
        Real(param_space_def["sharpness_growth"][0], param_space_def["sharpness_growth"][1], name="sharpness_growth"),
        Real(param_space_def["decay"][0], param_space_def["decay"][1], name="decay"),
        Integer(param_space_def["max_freq"][0], param_space_def["max_freq"][1], name="max_freq"),
        Real(param_space_def["weight_exponent"][0], param_space_def["weight_exponent"][1], name="weight_exponent"),
        Real(param_space_def["freq_sharpness_exp"][0], param_space_def["freq_sharpness_exp"][1], name="freq_sharpness_exp"),
    ]

    # will store intermediate results for post-hoc analysis
    eval_records = []
    call_counter = {"n": 0}
    timing_info = {"start": None}

    # objective function for gp_minimize (we minimize)
    def objective_func(x):
        call_counter["n"] += 1

        # Startzeit merken
        if timing_info["start"] is None:
            timing_info["start"] = time.time()

        # vergangene Zeit berechnen
        elapsed = time.time() - timing_info["start"]
        avg_time = elapsed / call_counter["n"]
        remaining = avg_time * (n_calls - call_counter["n"])
        eta_min = int(remaining // 60)
        eta_sec = int(remaining % 60)

        print(f"[Bayesian Optimization] Iteration {call_counter['n']} / {n_calls} "
            f"(ETA: {eta_min}m {eta_sec}s)")

        # map x to param dict
        names = ["sharpness_base", "sharpness_growth", "decay", "max_freq",
                "weight_exponent", "freq_sharpness_exp"]
        params = default_params.copy()
        params.update({n: v for n, v in zip(names, x)})
        params["max_freq"] = int(round(params["max_freq"]))

        # run analysis (this is the expensive call)
        results_df = analyze_folder_global_null(folder_path, params,
                                                num_sequences=num_sequences, seed=seed)

        if results_df.empty:
            return 1e6

        col_map = {
            "diff": f"diff_real_{null_model}",
            "zscore": f"zscore_{null_model}",
            "pval": f"pval_{null_model}"
        }
        col = col_map[objective]
        if col not in results_df.columns:
            return 1e5

        vals = results_df[col].dropna().values
        if len(vals) == 0:
            return 1e5

        if objective in ("diff", "zscore"):
            agg = np.median(vals)
            score = -float(agg)
        else:  # pval
            agg = np.median(vals)
            score = float(agg)

        eval_records.append({**params, f"agg_{objective}_{null_model}": float(agg)})

        return score

    # run Bayesian optimization
    res = gp_minimize(objective_func, sk_space, n_calls=n_calls, random_state=seed)

    # collect evaluated params into DataFrame
    summary_eval_df = pd.DataFrame(eval_records)

    # determine best set according to chosen objective
    agg_col_name = f"agg_{objective}_{null_model}"
    if agg_col_name in summary_eval_df.columns and not summary_eval_df.empty:
        if objective in ("diff", "zscore"):
            best_idx = summary_eval_df[agg_col_name].idxmax()
        else:  # pval
            best_idx = summary_eval_df[agg_col_name].idxmin()
        best_row = summary_eval_df.loc[best_idx]
    else:
        best_row = None

    return summary_eval_df, best_row, res

# ================================
# Optimierung + Sammeln der Sequenz-Werte
# ================================

def optimize_and_collect(folders, n_calls=30, num_sequences=50, seed=42, objective="pval"):
    all_seq_results = []

    for folder in folders:
        for null_model in ["expon", "uniform", "empirical"]:
            print("==============================")
            print(f"Starte Optimierung: Folder={folder}, Nullmodell={null_model}, Objective={objective}")

            # 1. Bayesian Optimization laufen lassen
            summary_eval_df, best_row, res = bayesian_search_global_null(
                folder_path=folder,
                n_calls=n_calls,
                num_sequences=num_sequences,
                seed=seed,
                objective=objective,
                null_model=null_model
            )

            if best_row is None:
                print("⚠️ Keine optimalen Parameter gefunden.")
                continue

            # 2. Beste Parameter extrahieren
            param_cols = ["sharpness_base","sharpness_growth","decay","max_freq","weight_exponent","freq_sharpness_exp"]
            best_params = {col: best_row[col] for col in param_cols}
            best_params.update({"norm_x":1.0, "output_max":1.0})

            print("Beste Parameterkombination:")
            print(best_params)

            # 3. Sequenzen mit besten Parametern analysieren
            df_seq = analyze_folder_global_null(
                folder_path=folder,
                params=best_params,
                num_sequences=num_sequences,
                seed=seed
            )

            if df_seq.empty:
                continue

            # 4. Werte pro Sequenz speichern
            for idx, row in df_seq.iterrows():
                record = {
                    "folder": folder.split("/")[-1],
                    "filename": row["filename"],
                    "null_model": null_model,
                    "pval": row[f"pval_{null_model}"],
                    "zscore": row[f"zscore_{null_model}"],
                    "diff": row[f"diff_real_{null_model}"]
                }
                # gefundene Parameter mit aufnehmen
                for p in param_cols:
                    record[p] = best_params[p]
                all_seq_results.append(record)

    return pd.DataFrame(all_seq_results)

# ================================
# Boxplots zeichnen
# ================================

def plot_all_boxplots(results_df, metrics=("pval","zscore","diff")):
    folders = results_df["folder"].unique()

    for folder in folders:
        subset = results_df[results_df["folder"] == folder]

        plt.figure(figsize=(15,5))
        for i, metric in enumerate(metrics, 1):
            plt.subplot(1, len(metrics), i)
            sns.boxplot(data=subset, x="null_model", y=metric, order=["expon","uniform","empirical"])
            plt.title(f"{metric} – {folder}")
            plt.xlabel("Nullmodell")
            plt.ylabel(metric)
        plt.tight_layout()
        plt.show()

# ================================
# Beispiel-Aufruf
# ================================
folders = [
    "/Users/emilschmiedl/Desktop/Proximity-Funktion/data/Kwa-Sounds Scorpaena spp",
    "/Users/emilschmiedl/Desktop/Proximity-Funktion/data/Bat Carollia perspicillata (C_persp)"
]

seq_results_df = optimize_and_collect(folders, n_calls=50, num_sequences=200, seed=42, objective="pval")

if not seq_results_df.empty:
    plot_all_boxplots(seq_results_df)
    seq_results_df.to_csv("seq_level_results.csv", index=False)
    print("Sequenz-Level-Ergebnisse in 'seq_level_results.csv' gespeichert.")
